# Exercise Session 3: Statistical Inference

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

## Exercise: Mass Estimation from Position Measurements in 1D Brownian Motion

A Brownian particle of mass $m_\star$ moves in one dimension in a fluid characterized by a friction coefficient $\gamma$ and a temperature $T$. Its velocity $v(t)$ can be modelled as an Ornstein-Uhlenbeck process, that is as a random variable varying in time according to the stochastic differential equation ([SDE](https://en.wikipedia.org/wiki/Stochastic_differential_equation)):
$$
m_\star \, dv(t) = - \gamma v(t) \, dt + \sqrt{2 \gamma k_B T} \, dW(t),
$$
where $k_B = 1.380649×10^{−23} \mathrm{J/K}$ is the Boltzmann constant and $W(t)$ is a [Wiener process](https://en.wikipedia.org/wiki/Wiener_process). The solution of the above SDE is a random function of time following a Gaussian distribution, which at stationarity (i.e. $t \to \infty$) becomes the well-known [Maxwell-Boltzmann distribution](https://en.wikipedia.org/wiki/Maxwell%E2%80%93Boltzmann_distribution).

In practice, we are interested in a model for the position of the particle in time, since the position of the Brownian particle is what we measure directly. Assuming that the velocity is stationary, for the stochastic process above the displacement $\Delta x$ occurring in the time interval $\Delta t$ is also a random Gaussian variable, with zero mean and variance given by the formula:
$$
\mathrm{Var}\{ \Delta x \} = \sigma^2(m_\star, \gamma, T, \Delta t) =  \frac{2 k_B T}{\gamma} \biggl[ \Delta t - \frac{m_\star}{\gamma} \Bigl( 1 - e^{-\gamma \Delta t / m_\star} \Bigr) \biggr].
$$

Given these premises, in this exercise we will consider the problem of inferring the true mass $m_\star$ from a series of position measurements realized when the system is stationary, given $\gamma$ and $T$.

### Task 1: simulating the Brownian motion, i.e. generating the dataset of displacements

You observe the particle's position at equally spaced times $t_0, t_1, \dots, t_n$, with $t_i - t_{i-1} = \Delta t$. Using the following values for the parameters of the problem

In [ ]:
# Parameters
m_star = 5.0e-10
gamma = 1.0e-8
temp = 300.0 # Temperature T
dt = 0.01
n = 200

simulate $n$ displacements $\Delta x_1, \dots, \Delta x_n$ as i.i.d. (independent and identically distributed) Gaussian variables with zero mean and variance $\sigma^2(m_\star, \gamma, T, \Delta t)$. Then, construct the corresponding trajectory. For simplicity, assume that $x_0 = 0$.

**1.1** Code the function $\sigma^2(m, \gamma, T, \Delta t)$.

In [ ]:
def displacement_var(m, gamma, temp, dt, kB = 1.380649e-23):
  result = 2 * kB * temp / gamma * (dt - m/gamma * (1 - np.exp(-gamma * dt / m)))
  return result

**1.2** Write a function that takes $m$, $\gamma$, $T$, $\Delta t$ and $n$ as inputs and generates the dataset of displacements as a vector of length $n$.

In [ ]:
def generate_dataset(m, gamma, temp, dt, n):
  sigma2 = displacement_var(m,gamma, temp, dt)
  return np.random.normal(0.0, np.sqrt(sigma2), n)

**1.3** Generate a dataset.

In [ ]:
dataset = gener

Ellipsis

**1.4** As a sanity check, plot its histogram and the Gaussian p.d.f. $\mathcal{N}(0, \sigma^2(m_\star, \gamma, T, \Delta t))$ from which the dataset should be drawn. Use reasonable limits for the $x$-axis.

*Hint*: To plot the histogram you can use ```hist``` from ```matplotlib.pyplot``` (check [documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html)). Set ```density = True``` to have the histogram represent a p.d.f. (remember that large $y$-values are expected when your p.d.f. is narrow). To plot the theoretical Gaussian you can employ the function ```norm``` from ```scipy.stats```, which has already been imported at the beginning of this notebook (check [documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)).

In [ ]:
...

Ellipsis

**1.5** Plot the trajectory corresponding to your dataset.

*Remark*: The plot will display **one realization** of the particle trajectory.

In [ ]:
...

Ellipsis

### Task 2: estimation of $m_\star$ through maximum likelihood

In this section, we implement the maximum likelihood estimator (MLE) for the mass, and use it to estimate $m_\star$ for the dataset that you generated.

**2.1** Prove that MLE $\hat{m}$ is the solution to the following equation
$$
\sigma^2(\hat{m}, \gamma, T, \Delta t) = S_n^2
$$
where $S_n^2 = \frac{1}{n} \sum_{i = 1}^n (\Delta x_i)^2$ is the (biased) estimator for the variance.

*Hint*: It is useful to maximise the log-likelihood instead of the likelihood to convert products into sums (recall that logs do not alter the position of maxima).

**2.2** Show that the previous equation can be cast as a fixed-point equation in $\hat{m}$, i.e. an equation of the form $\hat{m} = F(\hat{m})$, with $F$ to be determined.

*Remark*: **What is the fixed-point iteration method?**

The fixed-point iteration method is a simple numerical technique to solve equations of the form $x = F(x)$. The idea is:
1. Start with an initial guess $x_0$.
2. Compute a new value by plugging into the function: $x_{k+1} = F(x_k)$.
3. Repeat until the sequence $\{ x_k \}$ stops changing (converges).

In particular, regarding point 3., one usually needs to define a convergence criterion. A standard choice is to compute at each step the absolute difference $|x_{k+1} - x_{k}|$ and to claim convergence when this is smaller than a conventionally small enough tolerance. Then, $x_{k+1}$ is taken as the approximate solution to the fixed-point equation.

However, convergence is not always guaranteed. If the function $F(x)$ is well-behaved (specifically, if $|F'(x)| < 1$ near the solution), then the iteration will converge to the fixed point $x^*$ that satisfies $x^* = F(x^*)$.

In our problem, the MLE equation for $\hat{m}$ cannot be solved in closed form, but we were able to rewrite it as a fixed-point equation. This allows us to approximate $\hat{m}$ by repeatedly applying $F$ until the value stabilizes.

**2.3** Assuming that $F(m)$ is well-behaved, write a function that computes the fixed point of the above equation using the fixed-point iteration method.

*Hint*: A proper function must have an optional initial value for the mass, an optional value for the tolerance and an optional maximum number of iterations (for safety). Additionally, in the case of no convergence, it should raise an error or simply return ```None``` after printing a warning message.

In [ ]:
def fixed_point_estimator():
    ...

**2.4** Compute $\hat{m}$ (via fixed-point iteration) on the dataset that you generated. Then plot the log-likelihood of your dataset as a function of the parameter $m$ together with the two vertical lines $m = \hat{m}$ and $m = m_\star$. Add a legend for clarity.

*Hint*: Take a reasonable initial value for $m$ (that means positive). Play with the tolerance value and the maximum number of iterations in your fixed-point iteration in order to get the best approximation of the MLE. Use a log scale for the $x$-axis. It is helpful to use different line-styles for the two vertical lines.

*Remark*: Again, the point that you obtain via fixed-point iteration is an **approximation** for the point maximizing the (log-)likelihood, namely the MLE. The latter in turn is what we use to estimate the ground-truth value of the mass. In general, there is no reason to expect the MLE to give a "very good" estimate of the true parameter (we will come back to this point in the last task of the exercise).

In [ ]:
...

Ellipsis

### Task 3: likelihood flattening in the diffusive regime

For an Ornstein-Uhlenbeck velocity process we can distinguish two interesting regimes:
- If $\gamma \Delta t / m_\star << 1$, the motion is *ballistic* (dominated by inertia).
- If $\gamma \Delta t / m_\star >> 1$, the motion is *diffusive* (dominated by diffusion).

**3.1** Check that the values for $m_\star$, $\gamma$ and $\Delta t$ considered until now do <u>not</u> correspond to a diffusive regime.

**3.2** Replace the previously employed value for $m_\star$ with

In [ ]:
m_star2 = 5.0e-15

and repeat the same check. Are you still out of the diffusive regime with this new value of $m_\star$?

**3.3** Using the same $\gamma$, $T$, $\Delta t$ and $n$ as before, generate a dataset of displacements using the new value for the ground-truth mass. Then plot the corresponding log-likelihood as a function of $m$. Do you observe any difference w.r.t. the last plot? Is maximizing the (log-)likelihood still a reasonable approach in this situation?

In [ ]:
...

Ellipsis

**3.4** Compute (via fixed-point iteration) the MLE on the dataset generated with the new value of $m_\star$. Play with the tolerance value and the maximum number of iterations. Are you able to obtain a reasonable estimate, that is one of the same order of magnitude of the true value?

In [ ]:
...

Ellipsis

**3.5** Find an explanation for the results that you have just found.

*Hint*: One way is to rewrite $\sigma^2(m_\star, \gamma, T, \Delta t)$ for $\gamma \Delta t / m_\star >> 1$.

### Task 4: dependence on the dataset size

Let's return to the initial setting of parameters. What happens if we have a small number of observations?

**4.1** Generate a dataset of size $n = 20$.

In [ ]:
...

Ellipsis

**4.2** Repeat task 2.4 on this new dataset. What happens to the MLE?

In [ ]:
...

Ellipsis

**4.3** Find an explanation for the results that you have just found.

*Hint*: The law of large numbers is involved.